# Resultados S vs Laura MCTS

Este notebook resume las 50 partidas jugadas entre `Group S` y `Laura MCTS`, alternando quien inicia. El objetivo es analizar resultado, efecto del primer jugador y costo computacional.

**Configuracion:**

- `Group S`: Trial-Based Online Policy Improvement con rollouts heuristicos, `TRIALS = 50`.
- `Laura MCTS`: agente MCTS con `num_simulations = 100`.
- Partidas: 50.
- Alternancia: S inicia 25 partidas y Laura inicia 25 partidas.


## Carga de datos

El archivo `s_vs_laura_50_results.csv` tiene dos filas por partida: una desde la perspectiva de S y otra desde la perspectiva de Laura. Para evitar duplicar conteos, los resumenes por partida usan solo las filas donde `focal_side == 'agent_1'`.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
CSV_PATH = PROJECT_ROOT / 's_vs_laura_50_results.csv'
FIG_DIR = PROJECT_ROOT / 'figuras_entrega'

df = pd.read_csv(CSV_PATH)
games = df[df['focal_side'] == 'agent_1'].copy().sort_values('game_id')

print('Filas del CSV:', len(df))
print('Partidas reales:', games['game_id'].nunique())
print('Tiempo total medido (min):', round(games['total_game_time'].sum() / 60, 2))

games.head()

## Resumen global

| Resultado | Partidas | Porcentaje |
| --- | ---: | ---: |
| Gana S | 43 | 86% |
| Gana Laura | 5 | 10% |
| Empate | 2 | 4% |

S tuvo un resultado global claramente favorable: gano 43 de 50 partidas.

In [ ]:
winner_summary = (
    games['winner_agent']
    .value_counts()
    .rename_axis('winner_agent')
    .reset_index(name='partidas')
)
winner_summary['porcentaje'] = winner_summary['partidas'] / len(games)
winner_summary

## Resultado segun quien inicia

| Quien inicia | Partidas | Gana S | Gana Laura | Empates | Win rate S |
| --- | ---: | ---: | ---: | ---: | ---: |
| Laura inicia | 25 | 24 | 0 | 1 | 96% |
| S inicia | 25 | 19 | 5 | 1 | 76% |

El resultado mas fuerte para S ocurre cuando Laura inicia: S gana 24 de 25 partidas. Cuando S inicia, Laura consigue sus 5 victorias, pero S sigue ganando la mayoria.

In [ ]:
by_starter = (
    games.groupby(['starter', 'winner_agent'])
    .size()
    .unstack(fill_value=0)
)
by_starter

## Grafica 1: resultados alternando inicio

La primera grafica muestra el resultado global y luego separa el resultado segun quien inicia.

![Resultados S vs Laura](figuras_entrega/10_s_vs_laura_resultados.png)

## Tiempos y duracion

| Metrica | Valor |
| --- | ---: |
| Tiempo total medido | 21.15 min |
| Tiempo promedio por partida | 25.38 s |
| Movimientos promedio por partida | 21.34 |

S tarda mas por jugada que Laura, porque cada decision ejecuta rollouts heuristicos. Aun asi, el resultado competitivo fue favorable para S con `TRIALS = 50`.

In [ ]:
timing_summary = pd.DataFrame({
    'metrica': [
        'tiempo_total_min',
        'tiempo_promedio_partida_s',
        'movimientos_promedio',
    ],
    'valor': [
        games['total_game_time'].sum() / 60,
        games['total_game_time'].mean(),
        games['num_moves'].mean(),
    ],
})
timing_summary

## Grafica 2: costo computacional

La segunda grafica compara el tiempo promedio por jugada de cada agente y muestra la duracion de cada partida.

![Tiempos S vs Laura](figuras_entrega/11_s_vs_laura_tiempos.png)

## Lectura final

Los resultados sugieren que la politica de S es fuerte frente a Laura MCTS en esta configuracion. La ventaja no viene gratis: S consume mas tiempo por jugada. Por eso, si se quisiera usar este agente en un torneo con limite estricto de tiempo, la mejora natural seria ajustar dinamicamente los trials o cortar temprano acciones claramente inferiores.

## Regenerar graficas

Esta celda es opcional. Solo hace falta ejecutarla si se modifica el CSV o si se quieren volver a exportar las imagenes.

In [ ]:
REGENERAR_GRAFICAS = False

if REGENERAR_GRAFICAS:
    import runpy
    runpy.run_path('generar_graficas_s_vs_laura.py')